# Phase 3.2 — Run 2B: Blind Diffusion Ablation

Runs 40 FL rounds with diffusion ON but **no PoC gating** — all clients (including Byzantine) get synthetic data.

> Select **GPU T4 ×2** and enable **Internet** before running.

In [ ]:
# Cell 1: Install Dependencies
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip install -q wfdb flwr web3 py-solc-x pyyaml
!pip install -q --no-deps diffusers accelerate
!pip install -q safetensors huggingface-hub filelock regex requests importlib-metadata

from solcx import install_solc
install_solc('0.8.19')
print('\u2705 Solidity compiler installed')

!npm install -g ganache 2>/dev/null && echo '\u2705 Ganache installed'
print('\u2705 All dependencies ready!')

In [ ]:
# Cell 2: Copy Code to Working Directory
import shutil, os, glob

DATASET_PATHS = glob.glob('/kaggle/input/*')
print(f'Datasets: {DATASET_PATHS}')

dataset_path = DATASET_PATHS[0] if DATASET_PATHS else None
if dataset_path is None:
    raise FileNotFoundError('No dataset found. Click Add Data and upload your zip.')

# Find repo root (handles nested zip extraction)
repo_root = None
for root, dirs, files in os.walk(dataset_path):
    if root.count(os.sep) - dataset_path.count(os.sep) > 3:
        break
    if 'main.py' in files and 'core' in dirs:
        repo_root = root
        break

if repo_root is None:
    # Maybe main.py is directly in dataset_path
    if os.path.isfile(os.path.join(dataset_path, 'main.py')):
        repo_root = dataset_path
    else:
        # Check one level deep
        for item in os.listdir(dataset_path):
            candidate = os.path.join(dataset_path, item)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'main.py')):
                repo_root = candidate
                break

if repo_root is None:
    raise FileNotFoundError(f'Cannot find main.py. Contents: {os.listdir(dataset_path)}')

print(f'Repo root: {repo_root}')

WORK = '/kaggle/working'
for item in os.listdir(repo_root):
    if item.startswith('.'): continue
    src = os.path.join(repo_root, item)
    dst = os.path.join(WORK, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

os.chdir(WORK)
assert os.path.isfile('main.py'), 'main.py not found!'
assert os.path.isdir('core'), 'core/ not found!'
print(f'\u2705 Code copied. Files: {sorted(os.listdir("."))[:15]}')

In [ ]:
# Cell 3: Start Ganache + Deploy Contract
import subprocess, time

ganache_proc = subprocess.Popen(
    'NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

if ganache_proc.poll() is not None:
    ganache_proc = subprocess.Popen(
        'NODE_OPTIONS=--max-old-space-size=8192 ganache-cli -p 8545 --accounts 15 --deterministic --gasLimit 30000000',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(10)

print(f'Ganache PID: {ganache_proc.pid}, running: {ganache_proc.poll() is None}')

!PYTHONPATH=/kaggle/working python core/deploy_contract.py
print('\u2705 Blockchain ready!')

In [ ]:
# Cell 4: Download + Preprocess + Partition Data
!rm -rf data/* checkpoints/*
!PYTHONPATH=/kaggle/working python core/download_data.py
!PYTHONPATH=/kaggle/working python core/preprocess_data.py
!PYTHONPATH=/kaggle/working python core/partition_data.py
print('\u2705 Data ready!')

In [ ]:
# Cell 5: RUN BLIND ABLATION (40 rounds, ~9 hours)
!PYTHONPATH=/kaggle/working python main.py --mode blind-ablation

In [ ]:
# Cell 6: Display Results
import numpy as np
try:
    data = np.load('checkpoints/results_2B_blind.npy', allow_pickle=True).item()
    f1 = data['final_f1']
    names = ['Normal', 'LBBB', 'RBBB', 'APB', 'PVC']
    print('Per-Class F1 Scores:')
    for n, s in zip(names, f1):
        print(f'  {n}: {s:.4f}')
    print(f'  Mean: {np.mean(f1):.4f}')
except:
    print('Results not found yet.')